# Соколов Б.О. ИУ5-23М

# Лабораторная работа №3: Реализация алгоритма Policy Iteration.

## Задание:

На основе рассмотренного на лекции примера реализуйте алгоритм Policy Iteration для любой среды обучения с подкреплением (кроме рассмотренной на лекции среды Toy Text / Frozen Lake) из библиотеки Gym (или аналогичной библиотеки).

In [5]:
!pip install gym pygame

In [6]:
!pip install -q gymnasium

In [9]:
import numpy as np
from pprint import pprint
import gymnasium as gym


class PolicyIterationAgent:
    """
    Агент, обучающийся с помощью алгоритма Policy Iteration.
    """
    def __init__(self, env, gamma=0.99, theta=1e-6, max_iter=1000):
        self.env = env
        self.gamma = gamma
        self.theta = theta
        self.max_iter = max_iter

        # Размерности среды (нужно использовать unwrapped для доступа к P)
        self.n_states = env.unwrapped.observation_space.n
        self.n_actions = env.unwrapped.action_space.n
        self.actions = np.arange(self.n_actions)

        # Начальная равномерная случайная политика
        self.policy = np.full((self.n_states, self.n_actions), 1.0 / self.n_actions)
        # Функция ценности состояний
        self.V = np.zeros(self.n_states)

    def print_policy(self):
        """Вывод матрицы политики."""
        print('Стратегия (policy):')
        pprint(self.policy)

    def policy_evaluation(self):
        """Итеративное оценивание политики до сходимости."""
        for _ in range(self.max_iter):
            delta = 0
            V_new = np.zeros(self.n_states)
            for s in range(self.n_states):
                v = 0
                for a, action_prob in enumerate(self.policy[s]):
                    if action_prob == 0:
                        continue
                    for prob, next_state, reward, done in self.env.unwrapped.P[s][a]:
                        # Если переход завершает эпизод, будущая ценность = 0
                        v += action_prob * prob * (reward + (0 if done else self.gamma * self.V[next_state]))
                V_new[s] = v
                delta = max(delta, abs(v - self.V[s]))
            self.V = V_new
            if delta < self.theta:
                break
        return self.V

    def policy_improvement(self):
        """Улучшение политики на основе текущей функции ценности."""
        policy_stable = True
        new_policy = np.zeros_like(self.policy)
        for s in range(self.n_states):
            q = np.zeros(self.n_actions)
            for a in range(self.n_actions):
                for prob, next_state, reward, done in self.env.unwrapped.P[s][a]:
                    q[a] += prob * (reward + (0 if done else self.gamma * self.V[next_state]))

            # Выбираем действия с максимальным Q (с допуском на погрешность)
            best_actions = np.where(q >= np.max(q) - 1e-10)[0]
            new_policy[s, best_actions] = 1.0 / len(best_actions)

            if not np.array_equal(new_policy[s], self.policy[s]):
                policy_stable = False

        self.policy = new_policy
        return policy_stable

    def policy_iteration(self, max_policy_iter=100):
        """Основной цикл Policy Iteration."""
        print("Запуск Policy Iteration...")
        for i in range(max_policy_iter):
            self.V = self.policy_evaluation()
            stable = self.policy_improvement()
            print(f"Итерация {i+1}: политика {'стабильна' if stable else 'изменена'}")
            if stable:
                print(f"Политика сошлась за {i+1} итераций.")
                break
        else:
            print(f"Достигнут лимит итераций ({max_policy_iter}).")


def test_agent(agent):
    """Тестовый эпизод с текстовым выводом среды."""
    env_test = gym.make('CliffWalking-v1', render_mode='ansi')
    state, _ = env_test.reset()
    total_reward = 0
    done = False
    while not done:
        action_probs = agent.policy[state]
        action = np.random.choice(agent.actions, p=action_probs)
        next_state, reward, terminated, truncated, _ = env_test.step(action)
        total_reward += reward
        print(env_test.render())
        state = next_state
        done = terminated or truncated
    print(f"Эпизод завершён. Общая награда: {total_reward}")
    env_test.close()


def main():
    env = gym.make('CliffWalking-v1')
    env.reset(seed=42)  # для воспроизводимости
    agent = PolicyIterationAgent(env, gamma=0.99, theta=1e-6, max_iter=1000)
    print("Начальная политика:")
    agent.print_policy()
    agent.policy_iteration(max_policy_iter=100)
    print("\nОптимальная политика:")
    agent.print_policy()
    print("\nТестовый эпизод:")
    test_agent(agent)
    env.close()


if __name__ == '__main__':
    main()

Начальная политика:
Стратегия (policy):
array([[0.25, 0.25, 0.25, 0.25],
       [0.25, 0.25, 0.25, 0.25],
       [0.25, 0.25, 0.25, 0.25],
       [0.25, 0.25, 0.25, 0.25],
       [0.25, 0.25, 0.25, 0.25],
       [0.25, 0.25, 0.25, 0.25],
       [0.25, 0.25, 0.25, 0.25],
       [0.25, 0.25, 0.25, 0.25],
       [0.25, 0.25, 0.25, 0.25],
       [0.25, 0.25, 0.25, 0.25],
       [0.25, 0.25, 0.25, 0.25],
       [0.25, 0.25, 0.25, 0.25],
       [0.25, 0.25, 0.25, 0.25],
       [0.25, 0.25, 0.25, 0.25],
       [0.25, 0.25, 0.25, 0.25],
       [0.25, 0.25, 0.25, 0.25],
       [0.25, 0.25, 0.25, 0.25],
       [0.25, 0.25, 0.25, 0.25],
       [0.25, 0.25, 0.25, 0.25],
       [0.25, 0.25, 0.25, 0.25],
       [0.25, 0.25, 0.25, 0.25],
       [0.25, 0.25, 0.25, 0.25],
       [0.25, 0.25, 0.25, 0.25],
       [0.25, 0.25, 0.25, 0.25],
       [0.25, 0.25, 0.25, 0.25],
       [0.25, 0.25, 0.25, 0.25],
       [0.25, 0.25, 0.25, 0.25],
       [0.25, 0.25, 0.25, 0.25],
       [0.25, 0.25, 0.25, 0.25],
   

Вывод:

В ходе выполнения лабораторной работы был реализован алгоритм Policy Iteration для среды CliffWalking-v1 из библиотеки Gymnasium.
Алгоритм состоит из двух чередующихся этапов: оценивания стратегии (policy evaluation) и улучшения стратегии (policy improvement).

Результаты:

Начальная политика — равномерное случайное распределение вероятностей по всем 4 действиям для каждого состояния.

Алгоритм сошёлся за 5 итераций улучшения политики, после чего политика стабилизировалась.

Полученная оптимальная политика является детерминированной (вероятность 1 для наилучшего действия в каждом состоянии) и предписывает агенту двигаться вдоль края обрыва, не падая в него, и кратчайшим путём достигать целевой клетки.

В тестовом эпизоде агент, следуя оптимальной политике, прошёл маршрут длиной 13 шагов, получив итоговую награду –13, что соответствует оптимальному поведению в данной среде (каждый шаг даёт штраф –1, падение в обрыв – штраф –100).

Заключение:
Policy Iteration эффективно находит оптимальную стратегию для среды CliffWalking, демонстрируя сходимость за малое число итераций. Полученная политика позволяет агенту достигать цели без падения в обрыв с минимальным суммарным штрафом.